# Mock Interview Practice — Day 24
**FAANG / Top Product Company Level — Week 4, Day 3**

*Day 24 keeps the FAANG bar (Google, Amazon, Meta, Uber, Microsoft, Netflix, Adobe, Flipkart) with 2025–2026 trending topics: bitemporal restatements, exactly-once sinks, data-skipping layouts / liquid clustering, Snowflake Iceberg tables, inference-table drift monitoring, Data Vault PIT bridges. Every solution is runnable and self-verifying — run each cell and look for the `Q# OK` line.*

**18 questions · 8 categories**

| # | Category | Questions |
|---|---|---|
| 1 | SQL Advanced | Q1 sessionization + ordered funnel · Q2 bitemporal as-known-on revenue |
| 2 | Python — Loop/Function/Data Structures | Q3 consistent hashing ring · Q4 trie autocomplete top-k |
| 3 | PySpark Coding | Q5 out-of-order CDC MERGE · Q6 skewed join salting |
| 4 | Spark Optimization & Deep Spark | Q7 unified memory manager · Q8 Tungsten + whole-stage codegen |
| 5 | ETL — Failures, Late Data, Fault Tolerance | Q9 exactly-once sink · Q10 backfill + live increment · Q11 data-skipping layouts |
| 6 | Data Modeling | Q12 late-arriving dims / inferred members · Q13 Data Vault 2.0 PIT |
| 7 | Snowflake | Q14 Apache Iceberg + open catalogs · Q15 accelerator decision matrix |
| 8 | Monitoring, Logging & Orchestration | Q16 Jobs at 500-table scale · Q17 inference drift (PSI) · Q18 the 3 AM failure |

**Rules of engagement**: attempt out loud before opening a solution · 12-minute cap per question · log misses in the scorecard at the end.

## Q1 — SQL Advanced: Clickstream Sessionization + Ordered Funnel Drop-Off [FAANG]

**Interviewer (Amazon, retail analytics):** Raw clickstream lands as one row per event. Product wants sessions, plus how far each session got through the funnel `search → view_item → add_to_cart`.

**Requirements**
1. **Sessionize** users with a 30-minute inactivity gap — pure SQL (LAG + cumulative island flags), no UDFs.
2. Per session: start, end, duration, event count, distinct pages.
3. **Funnel stage per session** using *ordered-subsequence* semantics: the deepest prefix of `search → view_item → add_to_cart` that appears **in that order** in the session's event sequence. Explain why "session contains all 3 event types" (unordered) over-credits funnels.
4. Output per-session rows and a stage drop-off summary.

**Follow-ups they will ask**: where does this break at 10B events/day (window sort → shuffle, session-key cardinality), and when would you push sessionization into Structured Streaming instead of re-batching?

In [0]:
# Solution Q1 — gaps-and-islands sessionization + ordered (subsequence) funnel
from pyspark.sql import functions as F

events = spark.createDataFrame([
    ("u1", "2026-11-16 09:00:00", "search",      "home"),
    ("u1", "2026-11-16 09:02:00", "view_item",   "p1"),
    ("u1", "2026-11-16 09:03:00", "add_to_cart", "p1"),
    ("u1", "2026-11-16 10:45:00", "search",      "home"),   # 102-min gap -> NEW session
    ("u1", "2026-11-16 10:46:00", "view_item",   "p2"),
    ("u2", "2026-11-16 09:00:00", "view_item",   "p3"),    # browsed BEFORE searching
    ("u2", "2026-11-16 09:05:00", "search",      "home"),
    ("u2", "2026-11-16 09:06:00", "view_item",   "p1"),
    ("u3", "2026-11-16 09:00:00", "add_to_cart", "p9"),    # cart without any search first
    ("u3", "2026-11-16 09:01:00", "search",      "home"),
], "user string, ts string, event_type string, page string").withColumn("ts", F.to_timestamp("ts"))

events.createOrReplaceTempView("q1_events")

sessions = spark.sql("""
WITH ordered AS (                      -- global per-user event position
  SELECT user, ts, event_type, page,
         LAG(ts) OVER (PARTITION BY user ORDER BY ts) AS prev_ts,
         ROW_NUMBER() OVER (PARTITION BY user ORDER BY ts, event_type) AS rn
  FROM q1_events),
flagged AS (                            -- 30-min inactivity gap -> island start
  SELECT *, CASE WHEN prev_ts IS NULL
                     OR (unix_timestamp(ts) - unix_timestamp(prev_ts)) > 1800
                THEN 1 ELSE 0 END AS new_sess
  FROM ordered),
islands AS (                           -- cumulative sum = session id per user
  SELECT *, SUM(new_sess) OVER (PARTITION BY user ORDER BY rn) AS session_id
  FROM flagged),
s1 AS (SELECT user, session_id,
              MIN(CASE WHEN event_type = 'search' THEN rn END) AS p_search
       FROM islands GROUP BY user, session_id),
r2 AS (SELECT i.user, i.session_id, i.event_type, i.rn, s1.p_search
       FROM islands i JOIN s1 ON i.user = s1.user AND i.session_id = s1.session_id),
s2 AS (SELECT user, session_id,          -- step 2 must occur AFTER step 1's position
              MIN(CASE WHEN event_type = 'view_item' AND rn > p_search THEN rn END) AS p_view
       FROM r2 GROUP BY user, session_id),
r3 AS (SELECT r2.*, s2.p_view FROM r2 JOIN s2 ON r2.user = s2.user AND r2.session_id = s2.session_id),
s3 AS (SELECT user, session_id,
              MIN(CASE WHEN event_type = 'add_to_cart' AND rn > p_view THEN rn END) AS p_cart
       FROM r3 GROUP BY user, session_id),
base AS (SELECT user, session_id, MIN(ts) AS sess_start, MAX(ts) AS sess_end,
                COUNT(*) AS events, COUNT(DISTINCT page) AS pages
         FROM islands GROUP BY user, session_id)
SELECT b.user, CONCAT(b.user, '-s', b.session_id) AS session, b.sess_start, b.sess_end,
       unix_timestamp(b.sess_end) - unix_timestamp(b.sess_start) AS dur_sec,
       b.events, b.pages,
       CASE WHEN s1.p_search IS NOT NULL AND s2.p_view IS NOT NULL AND s3.p_cart IS NOT NULL THEN 3
            WHEN s1.p_search IS NOT NULL AND s2.p_view IS NOT NULL THEN 2
            WHEN s1.p_search IS NOT NULL THEN 1 ELSE 0 END AS funnel_stage
FROM base b
JOIN s1 ON b.user = s1.user AND b.session_id = s1.session_id
JOIN s2 ON b.user = s2.user AND b.session_id = s2.session_id
JOIN s3 ON b.user = s3.user AND b.session_id = s3.session_id
ORDER BY session""")
display(sessions)

funnel = sessions.groupBy("funnel_stage").count().orderBy("funnel_stage")
display(funnel)

expect = {"u1-s1": 3, "u1-s2": 2, "u2-s1": 2, "u3-s1": 1}
got = {r["session"]: r["funnel_stage"] for r in sessions.collect()}
assert got == expect, got
print("Q1 OK: 4 sessions from 10 events; u1 split at 10:45 (gap > 30 min)")
print("     u2 credits only search->view: its first view_item came BEFORE the search, but an")
print("     ordered subsequence search(2)->view(3) still exists, so stage 2 is correct")
print("     u3 stays at stage 1: a cart with no search before it must not count as a funnel hit")
print("     unordered 'contains all 3' would credit u3 stage 3 — that is the classic funnel bug")

## Q2 — SQL Advanced: Bitemporal Revenue — As-Known-On vs Final Restatements [FAANG]

**Interviewer (Netflix, finance data platform):** Order revenue arrives late and gets **restated**. Every fact row is bitemporal: it carries `event_date` (when the business event happened) and `arrival_date` (when the row landed in the lake). Analysts ask two different questions and most pipelines silently conflate them:

1. **As-known-on D**: what did we *believe* revenue per event day was on day D — needed to reproduce last quarter's dashboard or an old filing.
2. **Final**: the last restatement for every order, as of now.

**Requirements**
1. Produce a single query comparing revenue per `event_date` **as-known-on 2026-11-12** vs **final**.
2. Show the naive "filter `event_date <= D`, then take the latest row per order" answer and prove it overstates revenue (it applies restatements that had not arrived by D).
3. Explain: which question an SCD Type-2 *dimension* handles, and why *fact restatement* needs bitemporality (two valid-time axes) instead.

**Scale follow-up**: how do you serve arbitrary `as_of` dates without re-computing from raw history (hint: arrival-aligned snapshots / PIT fact versions)?

In [0]:
# Solution Q2 — bitemporal as-of dedup, final restatements, and the naive-overstatement trap
orders = spark.createDataFrame([
    ("o1", "2026-11-10", 100.0, "2026-11-10"),   # first report
    ("o1", "2026-11-10", 120.0, "2026-11-13"),   # restated UP on the 13th
    ("o2", "2026-11-10", 200.0, "2026-11-11"),
    ("o3", "2026-11-11",  50.0, "2026-11-11"),   # first report
    ("o3", "2026-11-11",  40.0, "2026-11-12"),   # restated DOWN
    ("o3", "2026-11-11",  45.0, "2026-11-14"),   # final
    ("o4", "2026-11-10", 300.0, "2026-11-15"),   # LATE arrival for an event on the 10th
], "order_id string, event_date string, amount double, arrival_date string")
orders.createOrReplaceTempView("q2_orders")

compare = spark.sql("""
WITH asof_d12 AS (                      -- latest row per order AMONG rows that arrived by D
  SELECT order_id, event_date, amount,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY arrival_date DESC) AS rn
  FROM q2_orders WHERE arrival_date <= '2026-11-12'),
as_known AS (SELECT order_id, event_date, amount FROM asof_d12 WHERE rn = 1),
final_ranked AS (                        -- latest row per order, arrival ignored
  SELECT order_id, event_date, amount,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY arrival_date DESC) AS rn
  FROM q2_orders),
final AS (SELECT order_id, event_date, amount FROM final_ranked WHERE rn = 1)
SELECT event_date,
       SUM(amount) FILTER (WHERE src = 'as_known') AS rev_as_known_1112,
       SUM(amount) FILTER (WHERE src = 'final')    AS rev_final
FROM (SELECT event_date, amount, 'as_known' AS src FROM as_known
      UNION ALL
      SELECT event_date, amount, 'final' FROM final)
GROUP BY event_date ORDER BY event_date""")
display(compare)

# naive: event_date filter but FINAL amounts — applies restatements that hadn't arrived
naive = spark.sql("""
SELECT SUM(amount) AS naive_rev FROM (
  SELECT amount, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY arrival_date DESC) AS rn
  FROM q2_orders WHERE event_date <= '2026-11-12') WHERE rn = 1""").collect()[0][0]

rows = {r["event_date"]: (r["rev_as_known_1112"], r["rev_final"]) for r in compare.collect()}
assert rows["2026-11-10"] == (300.0, 620.0), rows   # 100+200 as-known; 120+200+300 final
assert rows["2026-11-11"] == (40.0, 45.0), rows
true_as_known = 300.0 + 40.0
assert naive == 665.0 and naive > true_as_known, (naive, true_as_known)
print(f"Q2 OK: as-known-on-11/12 revenue = {true_as_known} vs final = {665.0}")
print(f"     naive (event_date filter + final amounts) reports {naive} — overstates by 325:")
print("     o4's 300 late-arriving order (event on 11-10, arrived 11-15) plus o1's +20 and o3's")
print("     +5 restatements all leak into a number that was supposedly 'as of 11/12'")
print("     SCD2 dimensions track attribute history at LOAD time; fact restatement is a second")
print("     valid-time axis (event vs arrival) — only bitemporal fact rows answer 'as-known-on D'")
print("     at scale: precompute arrival-aligned snapshots (PIT fact versions) per reporting date")

## Q3 — Python: Consistent Hashing Ring with Virtual Nodes [FAANG]

**Interviewer (Amazon, Dynamo lineage):** A cache tier shards keys with `hash(key) % N`. Every resize re-homes most keys, the cache miss storm takes the site down, and a dead node dumps its entire load onto one survivor. Rebuild routing the Dynamo way.

**Requirements**
1. Implement a consistent-hash **ring** with **virtual nodes** (md5-based hashing, ~150 vnodes per physical node): `add_node`, `remove_node`, `get_node`.
2. With 10k keys and 3 → 4 nodes, measure **how many keys move** under your ring vs the modulo scheme; state the theoretical bounds (~K/N vs ~K).
3. Show per-node key balance with and without vnodes, and explain: why a vnode count ≫ nodes fixes both balance and failover spread, and why Dynamo treats a *slow* node as a temporarily-removed one.

**Complexity to state**: O(log V) lookup per key with V = total vnodes.

In [0]:
# Solution Q3 — consistent hashing: vnodes, movement ~K/N, balance, failover spread
import hashlib, bisect, random
from collections import Counter

def h(key: str) -> int:
    return int(hashlib.md5(key.encode()).hexdigest()[:12], 16)

class Ring:
    def __init__(self, vnodes=500):   # Dynamo used ~150 for failover; more vnodes tighten balance
        self.vnodes = vnodes
        self.ring = []                 # sorted (hash, node) tuples
        self.nodes = set()
    def add_node(self, node):
        self.nodes.add(node)
        for v in range(self.vnodes):
            bisect.insort(self.ring, (h(f"{node}#vn{v}"), node))
    def remove_node(self, node):
        self.nodes.discard(node)
        self.ring = [e for e in self.ring if e[1] != node]
    def get_node(self, key):           # O(log V): first vnode clockwise from hash(key)
        i = bisect.bisect_left(self.ring, (h(key),))
        if i == len(self.ring):
            i = 0
        return self.ring[i][1]

KEYS = [f"key{i:05d}" for i in range(10_000)]

ring = Ring()
for n in ["n1", "n2", "n3"]:
    ring.add_node(n)
before = {k: ring.get_node(k) for k in KEYS}
ring.add_node("n4")
moved_ch = sum(before[k] != ring.get_node(k) for k in KEYS)

# modulo baseline: re-homes ~K keys on every resize
mod_moved = sum(h(k) % 3 != h(k) % 4 for k in KEYS)

# balance: ring w/ vnodes vs single vnode per node (plain consistent hashing)
load = Counter(ring.get_node(k) for k in KEYS)
plain1 = Ring(vnodes=1)
for n in ["n1", "n2", "n3"]:
    plain1.add_node(n)
load1 = Counter(plain1.get_node(k) for k in KEYS)

# failover: where a dead node's keys go — spread (vnodes) vs single survivor (1 vnode)
ring2 = Ring()
for n in ["n1", "n2", "n3"]:
    ring2.add_node(n)
n3_keys = [k for k in KEYS if ring2.get_node(k) == "n3"]
ring2.remove_node("n3")
failover = Counter(ring2.get_node(k) for k in n3_keys)

print(f"ring resize 3->4 nodes : {moved_ch:5d}/10000 keys moved ({moved_ch/100:.1f}%) — bound ~K/N = 25%")
print(f"modulo resize 3->4     : {mod_moved:5d}/10000 keys moved ({mod_moved/100:.1f}%) — ~everything")
print(f"load spread (vnodes=500): {dict(load)}   max/min = {max(load.values())/min(load.values()):.2f}")
print(f"load spread (vnodes=1)  : {dict(load1)}  max/min = {max(load1.values())/min(load1.values()):.2f}")
print(f"n3's {len(n3_keys)} keys on failure spread to {dict(failover)} (not one survivor)")

ratio_v = max(load.values()) / min(load.values())      # 500 vnodes per node
ratio_1 = max(load1.values()) / min(load1.values())    # 1 vnode per node
print(f"balance ratio: vnodes=500 -> {ratio_v:.2f}x | vnodes=1 -> {ratio_1:.2f}x")

assert moved_ch < 3_200 and mod_moved > 6_000          # ~K/N vs ~K
assert ratio_v < 1.15, ratio_v                         # vnodes: tight balance
assert ratio_v < ratio_1                               # ...and strictly better than 1 vnode/node
assert len(failover) == 2 and 0.30 < failover["n1"] / len(n3_keys) < 0.70
print("Q3 OK: O(log V) lookups; vnodes give ~K/N movement, even balance, and spread failover;")
print("     Dynamo 'gossip + suspicion': a slow node's vnodes are treated as removed, so its")
print("     load drains gradually instead of hard-failing — that's the 'temporarily removed' trick")

## Q4 — Python: Trie Autocomplete with Per-Node Top-K Ranking [FAANG]

**Interviewer (Google, search-as-you-type):** Serve the top-3 completions for every keystroke, ranked by query popularity, over a 100k-word corpus — without walking the whole subtree on each keystroke.

**Requirements**
1. Build a **Trie** where every node maintains its own top-k completion list (updated on insert), so `suggest(prefix)` costs O(len(prefix)) node visits plus k, independent of subtree size.
2. Handle popularity **updates** for an existing word (a node's cached top-k must not keep a stale entry).
3. Type out `n → ne → net → netf` and print suggestions at each keystroke; verify against a brute-force top-3 for every tested prefix.
4. Explain the write-amplification trade-off (each insert touches O(len) node top-k lists) and what production adds on top (personalization / recency re-ranking after the candidate fetch).

In [0]:
# Solution Q4 — trie with per-node cached top-k; insert, update, suggest, verify
import bisect, random

class TrieNode:
    __slots__ = ("children", "top")
    def __init__(self):
        self.children = {}
        self.top = []              # ascending by (-freq, word), capped at k

class Autocomplete:
    def __init__(self, k=3):
        self.root, self.k = TrieNode(), k
    def insert(self, word, freq):
        node = self.root
        entry = (-freq, word)
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
            node.top = [e for e in node.top if e[1] != word]   # drop stale entry on update
            bisect.insort(node.top, entry)
            if len(node.top) > self.k:
                node.top.pop()
    def suggest(self, prefix, node_visits=None):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return []
            if node_visits is not None:
                node_visits[0] += 1
            node = node.children[ch]
        return [w for _, w in node.top]

WORDS = {"netflix": 500, "netflix max": 260, "network": 300, "netgear": 90, "netscape": 50,
         "new": 420, "news": 400, "newspaper": 150, "next": 310, "nexus": 80,
         "neural": 60, "neural net": 40, "neuron": 55, "never": 200, "nest": 120}

ac = Autocomplete(k=3)
order = list(WORDS.items())
random.seed(9)
random.shuffle(order)
for w, f in order:
    ac.insert(w, f)
ac.insert("news", 450)             # popularity update AFTER first insert
WORDS["news"] = 450                 # keep the brute-force reference in sync

def brute(prefix, k=3):
    cand = sorted(((-f, w) for w, f in WORDS.items() if w.startswith(prefix)))
    return [w for _, w in cand[:k]]

for prefix in ["n", "ne", "net", "netf", "new", "news", "neural", "zz"]:
    visits = [0]
    got = ac.suggest(prefix, visits)
    assert got == brute(prefix), (prefix, got, brute(prefix))
    print(f"  type {prefix!r:10} -> {got}  ({visits[0]} node hops)")

assert ac.suggest("news") == ["news", "newspaper"]       # update re-ranked news to #1
print("Q4 OK: every keystroke answered in O(len(prefix)) hops regardless of subtree size;")
print("     popularity updates drop the stale cached entry before re-inserting; verified vs brute force;")
print("     trade-off: writes touch a top-k list on EVERY node of the path (O(len) inserts) — cheap for")
print("     words, not documents. Prod adds a re-rank layer (recency, personalization, session context)")
print("     AFTER this cheap candidate fetch.")

## Q5 — PySpark: Out-of-Order CDC — Sequencing, Deletes, and Stale Updates [FAANG]

**Interviewer (Uber):** CDC rows from two regions land out of order and duplicated (at-least-once export). Apply them to a Type-1 target without resurrecting deleted rows, replaying stale updates, or exploding the MERGE.

**Requirements**
1. Dedupe to the **latest LSN per key** (ties broken deterministically), so a stale `UPDATE` arriving after a newer one is suppressed and a `DELETE` wins over older updates.
2. A **single MERGE**: delete-when-matched-and-delete-op, update-when-matched, insert-when-not-matched-and-not-delete. State what `DELTA_MULTIPLE_MATCHES` / "multiple source rows matched" means and prove it by triggering the error once with the raw (non-deduped) feed.
3. Explain why processing the raw stream in *arrival* order instead of LSN order corrupts the target (lost updates, zombie rows) — and where this changes for a streaming `apply_changes` pipeline.

In [0]:
# Solution Q5 — LSN-ordered dedupe + guarded single MERGE + the multiple-matches error, live
import uuid
from pyspark.sql import functions as F, Window

run = uuid.uuid4().hex[:8]                      # fresh paths -> re-runnable any day
base = f"/Volumes/workspace/default/etl_test_volume/d24_q5_{run}"

cdc = spark.createDataFrame([                    # deliberately out of order + duplicated
    (3, "k1", "U", "NYC"),     # newest for k1
    (1, "k1", "I", "SF"),
    (2, "k1", "U", "LA"),     # STALE update, arrives after lsn 3
    (4, "k2", "U", "Chicago"),# stale: the delete at lsn 5 already won
    (5, "k2", "D", None),     # DELETE must win
    (6, "k3", "I", "Seattle"),
    (6, "k3", "I", "Seattle"), # exact duplicate row (at-least-once delivery)
], "lsn int, customer_id string, op string, city string")

# 1) latest event per key: LSN desc; op desc breaks exact-duplicate ties deterministically
latest = (cdc.withColumn("rn", F.row_number().over(
              Window.partitionBy("customer_id").orderBy(F.col("lsn").desc(), F.col("op").desc())))
            .filter("rn = 1").drop("rn"))

spark.createDataFrame([], "customer_id string, city string") \
     .write.format("delta").mode("overwrite").save(base)

latest.createOrReplaceTempView("q5_latest")
spark.sql(f"""
MERGE INTO delta.`{base}` AS t
USING q5_latest AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.op = 'D' THEN DELETE
WHEN MATCHED THEN UPDATE SET t.city = s.city
WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT (customer_id, city) VALUES (s.customer_id, s.city)
""")
display(spark.read.format("delta").load(base))

# 2) prove the classic failure: raw duplicated source keys vs MERGE
try:
    cdc.createOrReplaceTempView("q5_raw")
    spark.sql(f"""
    MERGE INTO delta.`{base}` AS t USING q5_raw AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET t.city = s.city
    WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT (customer_id, city) VALUES (s.customer_id, s.city)""")
except Exception as e:
    print(f"raw-feed MERGE -> {type(e).__name__}: {str(e)[:110]}...")

got = {r["customer_id"]: r["city"] for r in spark.read.format("delta").load(base).collect()}
assert got == {"k1": "NYC", "k3": "Seattle"}, got
print("Q5 OK: k1 kept lsn3 (stale lsn2 LA suppressed); k2 stayed deleted (stale lsn4 suppressed);")
print("     duplicate lsn6 collapsed; MERGE with duplicated source keys fails atomically — that")
print("     is why the dedupe MUST precede the MERGE. In arrival-order processing the LA update")
print("     would arrive after NYC and clobber it, and k2's Chicago update would resurrect a deleted")
print("     row. Streaming apply_changes/Auto CDC does this sequencing (sequence_by) for you.")

## Q6 — PySpark: Killing Skew — Salting vs Broadcast vs AQE [FAANG]

**Interviewer (Flipkart, Big Billion Days):** `orders` joins `cities` on `city_id`. One metro holds 40% of orders, the join's last stage has one straggler task, and the SLA dies with it.

**Requirements**
1. **Measure** the skew — rows per join key, and per-partition row counts of the shuffle.
2. Implement **salting**: big side gets ONE salt in `[0, S)`, the small side replicates to all S salts; join on `(key, salt)`; verify the salted result is row-for-row identical to the plain join.
3. Talk the alternatives precisely: broadcast join (when the dim fits driver+executor memory limits), AQE `skewJoin` (what it does — splits skewed partitions at reduce time — and what it needs: SortMergeJoin + size stats), and why pre-salting can *hide* skew from AQE so you pick one remedy, not both.

**Scale follow-up**: what changes when BOTH sides are terabyte-scale (answer: salting or AQE; broadcast is off the table).

In [0]:
# Solution Q6 — measure skew, salt the join, prove equality, place AQE/broadcast correctly
from pyspark.sql import functions as F

SALTS = 8
orders = spark.createDataFrame(
    [(f"o{i:05d}", "c1" if i < 2000 else f"c{i % 20 + 2:02d}") for i in range(5000)],
    "order_id string, city_id string")
cities = spark.createDataFrame(
    [("c1", "Mega City")] + [(f"c{i:02d}", f"City {i}") for i in range(2, 22)],
    "city_id string, city_name string")

key_skew = orders.groupBy("city_id").count().orderBy(F.desc("count"))
display(key_skew)

try:  # see the raw shuffle (some computes don't expose this conf — measurement stays valid)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
except Exception:
    pass
plain = orders.join(cities, "city_id")
dist_plain = (plain.groupBy(F.spark_partition_id().alias("partition"))
                   .agg(F.count("*").alias("rows")).orderBy(F.desc("rows")))

# salting: ONE salt per order row; dim replicated to every salt
orders_salted = orders.withColumn("salt", F.pmod(F.hash("city_id", "order_id"), F.lit(SALTS)))
cities_salted  = cities.withColumn("salt", F.explode(F.array(*[F.lit(i) for i in range(SALTS)])))
salted = orders_salted.join(cities_salted, ["city_id", "salt"]).drop("salt")
dist_salted = (salted.groupBy(F.spark_partition_id().alias("partition"))
                     .agg(F.count("*").alias("rows")).orderBy(F.desc("rows")))

print("plain join — top partitions (hot key concentrates in one reduce task):")
display(dist_plain.limit(5))
print("salted join — top partitions (hot key spread across SALTS reduce tasks):")
display(dist_salted.limit(5))
try:
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
except Exception:
    pass

# prove semantic equality: per-city totals identical, total rows identical
chk = (plain.groupBy("city_id").count().withColumnRenamed("count", "n_plain")
       .join(salted.groupBy("city_id").count().withColumnRenamed("count", "n_salted"), "city_id"))
assert chk.filter("n_plain <> n_salted").count() == 0
assert plain.count() == salted.count() == 5000
assert key_skew.first()["count"] == 2000
print("Q6 OK: c1 = 2000/5000 rows (40%) — one straggler task in the plain join; salting spread it")
print("     across 8 partitions with row-identical results. Broadcast kills the shuffle entirely")
print("     when the dim fits the 8/10GB limit; AQE skewJoin splits skewed SMJ partitions from size")
print("     stats at reduce time. Pre-salting makes every partition look 'normal', so AQE's skew")
print("     detection has nothing to see — choose ONE remedy, and with two TB-scale sides salting")
print("     or AQE are the only options.")

## Q7 — Spark Deep Dive: Unified Memory Manager, Spill, and GC [HARD]

**Interviewer (Meta, platform):** "Our join OOMs while 40% of storage memory sits unused." Explain Spark's executor memory model precisely enough to act on that sentence.

**Requirements**
1. Layout of executor heap memory: **reserved** (~300MB), the **unified** region governed by `spark.memory.fraction` (execution ⇄ storage, soft boundary), and **user** memory; what `spark.memory.storageFraction` really means (a *soft* eviction threshold, not a hard wall).
2. What spills and when: sort/agg/join buffers live in execution memory; when execution needs storage's pages, cached blocks are evicted LRU — but **execution memory is never evicted, it spills to disk**. Where you see it: Spark UI stage metrics (spill memory/disk), SQL metrics `spill size`.
3. GC: why Tungsten's binary/off-heap row format reduces GC pressure, what `spark.memory.offHeap.enabled` moves, and a config stack you'd try for a memory-pressured big shuffle join.

In [0]:
# Solution Q7 — memory layout with real numbers, eviction rules, spill diagnosis, GC + config stack

def conf_or(key, default):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

frac, stfrac = float(conf_or("spark.memory.fraction", "0.6")), float(conf_or("spark.memory.storageFraction", "0.4"))

M_MB = 16384.0                     # worked example: one 16g classic executor
reserved = 300.0
usable = M_MB - reserved
unified = usable * frac            # execution + storage share this, soft boundary
storage_soft = unified * stfrac   # eviction threshold, NOT a hard reservation
user_mem = usable * (1 - frac)
print(f"executor heap M              : {M_MB:.0f} MB")
print(f"reserved                      : {reserved:.0f} MB (JVM internals)")
print(f"spark.memory.fraction = {frac} -> unified (exec+storage): {unified:,.0f} MB")
print(f"spark.memory.storageFraction = {stfrac} -> soft storage threshold: {storage_soft:,.0f} MB")
print(f"user memory (UDFs, structs, etc.): {user_mem:,.0f} MB")

TALK_TRACK = """
WHY THE JOIN OOMs WITH 40% STORAGE IDLE
  Unified memory is a SOFT boundary: execution can borrow ALL free storage pages, and cached
  blocks get evicted (LRU) to release it — eviction is allowed, so idle storage never starves
  execution. If the OOM happens anyway it is usually because: (a) the request came from
  UNSAFE/off-heap or user memory (outside the unified region), (b) a single task's hash/sort
  buffer needed more than the max single grant, or (c) storage could not evict fast enough
  (blocks being written / non-cache). Then the executor OOMs rather than spills.

EVICTION vs SPILL (say this exactly)
  - Storage blocks: EVICTED (dropped, recomputed later if the RDD is used again).
  - Execution pages: NEVER evicted; SPILLED to local disk when memory can't grow.
  - Both live in the unified region; storageFraction is only an eviction *priority* line.

SEEING SPILL
  - Spark UI -> stage -> task: 'Spill (Memory)' / 'Spill (Disk)'; SQL metrics 'spill size'.
  - A join whose build side or sort overflows -> huge memory spill; more shuffle partitions
    shrink per-task buffers (spark.sql.shuffle.partitions), often the first cheap fix.

GC / TUNGSTEN
  - Tungsten stores rows in compact binary (on- or off-heap): no per-field Java objects, no
    object headers -> dramatically fewer GC'd objects, better cache locality.
  - spark.memory.offHeap.enabled + spark.memory.offHeap.size moves Tungsten pages out of the
    heap entirely -> less GC pressure, but you must reserve that RAM off-heap.

CONFIG STACK FOR A MEMORY-PRESSED SHUFFLE JOIN (in order of preference)
  1. broadcast the small side (removes the build-side buffer) — if it fits
  2. spark.sql.shuffle.partitions up (smaller per-task buffers, more tasks)
  3. enable off-heap + size it (Tungsten pages out of the GC'd heap)
  4. G1GC with region sizing; avoid tiny young gens with huge caches
  5. reduce input per task (salting / repartition, filter earlier, column prune)
"""
print(TALK_TRACK)
print(f"Q7 OK: layout math verified from live conf (fraction={frac}, storageFraction={stfrac});")
print("     eviction vs spill vs OOM decision tree stated; GC + config stack delivered")

## Q8 — Spark Deep Dive: Tungsten & Whole-Stage Codegen — and Where They Stop [HARD]

**Interviewer (Netflix, Spark-committer energy):** What IS Tungsten, what does whole-stage codegen fuse, and why does one Python UDF end the free lunch?

**Requirements**
1. Explain precisely: **Tungsten** binary row format + unsafe memory ops + cache locality; **whole-stage codegen** — the physical plan's chain of iterators (scan → filter → agg) compiled by Janino into ONE generated function, shown as `*(N)` in `explain`; virtualized calls instead of iterator boundaries.
2. **Run and read a real plan**: capture `EXPLAIN` of a filter+aggregate; on classic JVM Spark count whole-stage `*(N)` markers, and on Databricks note the native **Photon** operators that replace them; then add a Python UDF and show where the fused chain breaks (`BatchEvalPython` — serialization boundary, columnar-to-row conversion).
3. **Micro-benchmark** the same aggregation with native expressions, a Python UDF, and a pandas (vectorized) UDF; report the boundary cost and why pandas UDFs claw most of it back.
4. State which ops silently disable codegen (e.g. some complex types, lambdas in some APIs) and how to check: `spark.sql.codegen.wholeStage` / plan markers.

In [0]:
# Solution Q8 — real plans + markers + the native-vs-UDF-vs-pandasUDF boundary benchmark
import re, time
from pyspark.sql import functions as F, types as T

spark.range(0, 500_000).withColumn("k", F.col("id") % 1000) \
     .withColumn("x", (F.col("id") % 97 + 1) * 1.5) \
     .createOrReplaceTempView("q8_tx")

plan_native = spark.sql("""EXPLAIN SELECT k, SUM(x) FROM q8_tx WHERE x > 20 GROUP BY k""").collect()[0][0]
spark.udf.register("q8_fee", lambda v: float(v) * 0.99, "double")
plan_udf = spark.sql("""EXPLAIN SELECT k, SUM(q8_fee(x)) FROM q8_tx WHERE x > 20 GROUP BY k""").collect()[0][0]

def markers(p):                    # JVM Spark: '*(N)' whole-stage codegen ids
    return re.findall(r"\*\(\d+\)", p)

def photon_ops(p):                 # Databricks native engine: Photon* operators
    return re.findall(r"Photon\w+", p.split("== Photon Explanation ==")[0])

print(f"native plan : {len(markers(plan_native))} JVM whole-stage markers | "
      f"{len(photon_ops(plan_native))} Photon operators (vectorized native engine)")
print(f"UDF plan    : {len(markers(plan_udf))} JVM whole-stage markers | {len(photon_ops(plan_udf))} Photon operators")
print("UDF plan contains BatchEvalPython        :", "BatchEvalPython" in plan_udf)
print("UDF plan contains PhotonColumnarToRow    :", "PhotonColumnarToRow" in plan_udf)
print("native plan contains BatchEvalPython      :", "BatchEvalPython" in plan_native)

# boundary benchmark: same aggregation, three implementations
tx = spark.table("q8_tx").cache(); tx.count()
py_udf  = F.udf(lambda v: float(v) * 0.99, "double")
pd_udf  = F.pandas_udf(lambda s: s * 0.99, T.DoubleType())

def bench(expr, label):
    t0 = time.time()
    tx.groupBy("k").agg(F.sum(expr).alias("s")).agg(F.sum("s")).collect()
    dt = time.time() - t0
    print(f"{label:28s} {dt:6.2f}s")
    return dt

t_native = bench(F.col("x") * 0.99, "native expression")
t_py     = bench(py_udf(F.col("x")), "python UDF (row-wise)")
t_pd     = bench(pd_udf(F.col("x")), "pandas UDF (vectorized)")

print(f"Q8 OK: UDF overhead ~{t_py/max(t_native,1e-9):.0f}x native; pandas UDF ~{t_pd/max(t_native,1e-9):.1f}x")
print("     on classic JVM Spark the fused stages appear as '*(N)' whole-stage codegen; on modern")
print("     Databricks the native Photon engine runs scan->filter->agg columnar and fused. Either")
print("     way a Python UDF forces a row-based boundary (BatchEvalPython + columnar-to-row): rows")
print("     serialize to the Python worker one by one — the fused chain physically cannot cross it.")
print("     pandas UDFs batch the crossing (Arrow, zero-copy) so most of the boundary cost")
print("     disappears (measured above); full native execution stays expression-only.")

## Q9 — ETL: Exactly-Once Sinks — At-Least-Once Source + Idempotent Write [FAANG]

**Interviewer (Stripe):** Kafka delivers at-least-once; the ledger team demands exactly-once into Delta. Walk the semantics and prove them in code.

**Requirements**
1. State the three-legged stool: **replayable source** (Kafka offsets / file discovery), **idempotent sink** (re-applying the same batch changes nothing), **atomic commit** (Delta transaction log). Explain what breaks with any two legs (classic: crash AFTER table commit but BEFORE offset commit → replay → duplicates unless the sink is idempotent).
2. Implement the idempotent sink: `foreachBatch` MERGE on natural key (`event_id`). Replay the SAME micro-batch twice and show the table does not change; then deliver an overlapping batch (one re-sent event + one new) and show the update lands once.
3. Explain Databricks **idempotent writes** (`txnAppId` + `txnVersion` on the write): how the Delta commit silently de-dupes repeated batch versions — and why you fall back to MERGE-on-key when a batch mixes inserts with updates.

In [0]:
# Solution Q9 — crash-replay simulation: idempotent MERGE sink + overlapping batch
class Batch:                       # stands in for a Kafka micro-batch read by offset range
    def __init__(self, rows):
        self.df = spark.createDataFrame(rows, "event_id string, amount double")

run = uuid.uuid4().hex[:8]
path = f"/Volumes/workspace/default/etl_test_volume/d24_q9_{run}"
spark.createDataFrame([], "event_id string, amount double") \
     .write.format("delta").mode("overwrite").save(path)

def apply_batch(batch, batch_id):        # == streaming foreachBatch: idempotent on event_id
    batch.df.createOrReplaceTempView("q9_batch")
    spark.sql(f"""
    MERGE INTO delta.`{path}` AS t USING q9_batch AS s
    ON t.event_id = s.event_id
    WHEN MATCHED THEN UPDATE SET t.amount = s.amount
    WHEN NOT MATCHED THEN INSERT (event_id, amount) VALUES (s.event_id, s.amount)""")
    print(f"  applied batch {batch_id}: rows={spark.read.format('delta').load(path).count()}")

b1 = Batch([("e1", 25.0), ("e2", 40.0), ("e3", 35.0)])
apply_batch(b1, 1)
apply_batch(b1, 1)          # crash AFTER table commit, BEFORE offset commit -> full replay
b2 = Batch([("e3", 38.0), ("e4", 12.0)])   # e3 re-sent (overlap), e4 genuinely new
apply_batch(b2, 2)

tab = spark.read.format("delta").load(path)
rows = {r["event_id"]: r["amount"] for r in tab.collect()}
assert rows == {"e1": 25.0, "e2": 40.0, "e3": 38.0, "e4": 12.0}, rows
print("Q9 OK: replayed batch left the table untouched; overlapping e3 updated exactly once;")
print("     final table = 4 events, e3=38. at-least-once source + idempotent sink + atomic Delta")
print("     commit = effectively-once. Table-level idempotent writes (txnAppId/txnVersion) drop")
print("     whole repeated batch versions silently — use them for pure inserts; use MERGE-on-key")
print("     when one batch mixes inserts and updates, which a whole-batch version can't express.")

## Q10 — ETL: Backfill and Live Increment Without Duplicates or Regressions [HARD]

**Interviewer (Netflix):** A new pipeline goes live TODAY. The stream is already running; you must also backfill 2 years of history. Merge both without double-counting, without stopping the stream, and make every step re-runnable.

**Requirements**
1. The convergence pattern: backfill and increments both MERGE on a stable key with a **monotone version/arrival guard** (`s.version > t.version`), so a re-run of a STALE backfill can never overwrite newer streaming data. Explain why plain append double-counts and why an unguarded MERGE regresses.
2. The **checkpoint trap**: never reuse the streaming checkpoint for the backfill job; what `Trigger.AvailableNow` is for; why watermarks make the backfill path stateless-by-design (or must be bypassed).
3. Prove it: apply backfill → overlapping increment → RE-run the stale backfill; assert final counts, sums, and the newest value for the overlapping key all survive.

In [0]:
# Solution Q10 — version-guarded MERGE: backfill + increment converge; stale re-runs are inert
run = uuid.uuid4().hex[:8]
path = f"/Volumes/workspace/default/etl_test_volume/d24_q10_{run}"
spark.createDataFrame([], "month string, revenue double, version int") \
     .write.format("delta").mode("overwrite").save(path)

backfill = spark.createDataFrame(
    [(f"2025-{m:02d}", 1000.0 + 10 * m, 1) for m in range(1, 13)] +
    [(f"2026-{m:02d}", 2000.0 + 10 * m, 1) for m in range(1, 11)],
    "month string, revenue double, version int")            # 22 months, version 1
increment = spark.createDataFrame(
    [("2026-10", 4321.0, 2), ("2026-11", 4444.0, 2)],       # overlapping UPDATE + new month
    "month string, revenue double, version int")

def merge(df):
    df.createOrReplaceTempView("q10_batch")
    spark.sql(f"""
    MERGE INTO delta.`{path}` AS t USING q10_batch AS s
    ON t.month = s.month
    WHEN MATCHED AND s.version > t.version
      THEN UPDATE SET t.revenue = s.revenue, t.version = s.version
    WHEN NOT MATCHED
      THEN INSERT (month, revenue, version) VALUES (s.month, s.revenue, s.version)""")

merge(backfill);  print("  after backfill           :", spark.read.format("delta").load(path).count(), "rows")
merge(increment); print("  after streaming increment:", spark.read.format("delta").load(path).count(), "rows")
merge(backfill);  print("  STALE backfill re-run     :", spark.read.format("delta").load(path).count(), "rows")

tab = spark.read.format("delta").load(path)
rows = {r["month"]: r["revenue"] for r in tab.collect()}
assert len(rows) == 23 and rows["2026-10"] == 4321.0 and rows["2026-11"] == 4444.0
print("Q10 OK: 23 months, no duplicates; 2026-10 kept the INCREMENT value 4321 even after the stale")
print("     backfill re-ran (version guard suppressed the regression); plain append would have")
print("     double-counted every overlapping month. Checkpoint rules: the backfill is a separate")
print("     batch job — NEVER reuse the stream's checkpoint (offsets are meaningless for history);")
print("     Trigger.AvailableNow lets one streaming job sweep available data then exit, and the")
print("     watermark applies only to the stream's stateful ops, so the backfill stays stateless.")

## Q11 — ETL: Data Skipping — Partitioning vs Z-ORDER vs Liquid Clustering [HARD]

**Interviewer (Databricks):** A 100TB click table answers a selective filter by scanning almost everything. Stats say the wrong files are being pruned. Explain why, and how each layout strategy fixes or causes it.

**Requirements**
1. How Delta file skipping actually works: per-file min/max column stats + null counts in the transaction log; a file is a candidate only if its ranges overlap the predicate. Why HIGH-cardinality Hive partitioning backfires (millions of small files, partition-pruning only on equality, list explosion).
2. **Z-ORDER** (multi-dimensional sort, Wilson–Hilferty sampling, full rewrite cost, `OPTIMIZE ... ZORDER BY`) vs **liquid clustering** (`CLUSTER BY`, incremental clustering on write + background `OPTIMIZE`, no partition declaration, 2025+ default recommendation) — and when plain time-partitioning still wins (natural append-only time filters).
3. **Simulate**: 20 files × 500 rows under a random layout vs a clustered (sorted) layout; for predicate `country='IN' AND user_id BETWEEN 3000 AND 3100`, report files-to-scan under each — and say which strategy produced the cluster order.

In [0]:
# Solution Q11 — simulate per-file min/max stats: random vs clustered layout skip ratios
import random
random.seed(24)

N_FILES, ROWS = 20, 500
COUNTRIES = ["US", "IN", "UK", "DE"]
rows = [(f"u{i:04d}", COUNTRIES[i // 2500]) for i in range(N_FILES * ROWS)]   # contiguous blocks

random_rows = rows[:]; random.shuffle(random_rows)
clustered_rows = sorted(rows, key=lambda r: (r[1], r[0]))   # clustered on (country, user_id)

def files_to_scan(layout):
    stats = []
    for f in range(N_FILES):
        chunk = layout[f * ROWS:(f + 1) * ROWS]
        stats.append((min(u for u, _ in chunk), max(u for u, _ in chunk),
                      {c for _, c in chunk}))
    hits = [i for i, (lo, hi, cs) in enumerate(stats)
            if "IN" in cs and lo <= "u3100" and hi >= "u3000"]
    return hits, stats

hits_random,  stats_r = files_to_scan(random_rows)
hits_cluster, stats_c = files_to_scan(clustered_rows)

print("predicate: country='IN' AND user_id BETWEEN u3000 AND u3100")
print(f"random layout    : scans {len(hits_random):2d}/20 files ({100*len(hits_random)/20:.0f}% skipped... almost none)")
print(f"  sample file range u{stats_r[0][0][1:]}..u{stats_r[0][1][1:]} countries={sorted(stats_r[0][2])}")
print(f"clustered layout : scans {len(hits_cluster):2d}/20 files ({100*(1-len(hits_cluster)/20):.0f}% skipped)")

assert len(hits_random) >= 15 and len(hits_cluster) <= 2
if hits_cluster:
    i = hits_cluster[0]
    print(f"  clustered IN-file range: u{stats_c[i][0][1:]}..u{stats_c[i][1][1:]} countries={sorted(stats_c[i][2])}")
print("Q11 OK: clustering tightened each file's min/max ranges so the log stats prune 90% of files;")
print("     random layout leaves every file a candidate. Hive partitioning = coarse equality-only")
print("     pruning + small-file explosion at high cardinality; ZORDER = one-shot multi-column sort")
print("     (full rewrite) for mostly-static tables; liquid clustering (CLUSTER BY) clusters")
print("     incrementally on write and in the background — no partition choice, adaptive keys —")
print("     which is why it's the current default recommendation; keep time partitioning for")
print("     append-only tables whose queries are time-ranged.")

## Q12 — Data Modeling: Late-Arriving Dimensions — Inferred Members [FAANG]

**Interviewer (Walmart):** Click facts reference `customer_id` values the CRM dimension hasn't loaded yet — new stores open at midnight, feeds reconcile at noon. The fact-loading SLA is 4 AM and it does not wait for the CRM.

**Requirements**
1. Implement the **inferred-member** pattern: the fact load MERGE-inserts placeholder dim rows (`is_inferred = true`) keyed from the facts themselves; when the CRM row later lands, a MERGE backfills the placeholder and flips the flag.
2. Keep a `-1 UNKNOWN` member for genuinely invalid keys, and show the fact query that can never fail on a missing dim row.
3. Give the daily **audit**: how many facts sit on inferred members, which keys, oldest inferred age — so the business knows what it is reporting on.

**Why this beats waiting**: fact latency stays decoupled from dim latency; attribute enrichment is a later no-rewrite UPDATE.

In [0]:
# Solution Q12 — inferred-member dim: placeholder insert, CRM backfill, audit
run = uuid.uuid4().hex[:8]
dim_path = f"/Volumes/workspace/default/etl_test_volume/d24_q12_{run}"

# seeded dim: one real customer plus the -1 UNKNOWN member
spark.createDataFrame([("c100", "Ana", "CA", False), ("-1", "UNKNOWN", "UNKNOWN", False)],
                      "customer_id string, name string, region string, is_inferred boolean") \
     .write.format("delta").mode("overwrite").save(dim_path)

facts = spark.createDataFrame([
    ("f1", "c101", 42.0), ("f2", "c102", 10.0), ("f3", "c100", 7.0),
], "fact_id string, customer_id string, amount double")
facts.createOrReplaceTempView("q12_facts")

# 1) fact load: MERGE-insert inferred placeholders for every unseen fact key
spark.sql(f"""
MERGE INTO delta.`{dim_path}` AS t
USING (SELECT DISTINCT customer_id FROM q12_facts WHERE customer_id <> '-1') AS s
ON t.customer_id = s.customer_id
WHEN NOT MATCHED THEN INSERT (customer_id, name, region, is_inferred) VALUES (s.customer_id, NULL, NULL, true)""")
print("after fact load:")
display(spark.read.format("delta").load(dim_path))

# 2) CRM feed arrives later: backfill placeholders, insert genuinely new members
crm = spark.createDataFrame([("c101", "Beto", "TX"), ("c103", "Cara", "NY")],
                            "customer_id string, name string, region string")
crm.createOrReplaceTempView("q12_crm")
spark.sql(f"""
MERGE INTO delta.`{dim_path}` AS t USING q12_crm AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.is_inferred THEN UPDATE SET t.name = s.name, t.region = s.region, t.is_inferred = false
WHEN NOT MATCHED THEN INSERT (customer_id, name, region, is_inferred) VALUES (s.customer_id, s.name, s.region, false)""")
print("after CRM backfill:")
dim = spark.read.format("delta").load(dim_path)
display(dim)

# 3) audit: facts currently sitting on inferred members
audit = (facts.join(dim, "customer_id")
              .filter("is_inferred")
              .agg(F.count("*").alias("facts_on_inferred"),
                   F.concat_ws(",", F.collect_list("customer_id")).alias("keys"))
              .collect()[0])
rows = {r["customer_id"]: (r["name"], r["is_inferred"]) for r in dim.collect()}
assert rows == {"c100": ("Ana", False), "-1": ("UNKNOWN", False),
                "c101": ("Beto", False), "c102": (None, True), "c103": ("Cara", False)}, rows
assert audit["facts_on_inferred"] == 1 and audit["keys"] == "c102"
print(f"Q12 OK: audit = {audit['facts_on_inferred']} fact(s) on inferred member(s) {audit['keys']};")
print("     c101 backfilled + flag flipped; c103 arrived with no facts yet; c102 still inferred;")
print("     the -1 UNKNOWN member absorbs invalid keys so fact queries never break. Enrichment")
print("     is a later UPDATE — no fact rewrite — which is the whole point of the pattern.")

## Q13 — Data Modeling: Data Vault 2.0 — Satellites, Effectivity, and PIT Bridges [HARD]

**Interviewer (a bank's lakehouse program, you as lead modeler):** Two source systems load at different times. Leadership wants "customer profile **as of any date**, deterministic, reproducible". Build the vault slice that delivers it.

**Requirements**
1. Model `hub_customer` (hash keys), `sat_profile`, `sat_address` (dedup: latest load per hash per source), `link_order_customer`.
2. Build the **PIT (point-in-time) table**: for a set of as-of dates, each customer hash points to the satellite **version valid at that date** — and show the as-of query returning correct, independently-versioned attributes from each satellite (including a hash that exists in only one satellite).
3. Explain the DV load discipline: insert-only hubs/links/sats (no updates at load), why satellites load in parallel without touching each other, and where the PIT table replaces the classic star join for reproducibility.

In [0]:
# Solution Q13 — vault satellites + effectivity + PIT bridge + as-of query
from pyspark.sql import functions as F, Window

hub = spark.createDataFrame([("h1", "c-100"), ("h2", "c-200"), ("h3", "c-300")],
                            "customer_hash string, src_cust_id string")
sat_profile = spark.createDataFrame([                 # one satellite, own load rhythm
    ("2026-01-01", "h1", "Alice", "active"),
    ("2026-01-03", "h1", "Alice", "inactive"),      # restated on the 3rd
    ("2026-01-01", "h2", "Bob",   "active"),
], "load_date string, customer_hash string, name string, status string")
sat_address = spark.createDataFrame([                 # DIFFERENT satellite, different rhythm
    ("2026-01-01", "h1", "SF"),
    ("2026-01-04", "h1", "LA"),
    ("2026-01-02", "h3", "Seattle"),               # h3 has NO profile rows at all
], "load_date string, customer_hash string, city string")
link = spark.createDataFrame([("o1", "h1"), ("o2", "h2"), ("o3", "h3")],
                              "order_id string, customer_hash string")
for name, df in [("q13_hub", hub), ("q13_prof", sat_profile),
                 ("q13_addr", sat_address), ("q13_link", link)]:
    df.createOrReplaceTempView(name)

pit = spark.sql("""
WITH dates AS (SELECT explode(array('2026-01-01','2026-01-03','2026-01-05')) AS as_of),
pro_asof AS (
  SELECT p.customer_hash, d.as_of, p.name, p.status,
         row_number() OVER (PARTITION BY p.customer_hash, d.as_of ORDER BY p.load_date DESC) AS rn
  FROM q13_prof p CROSS JOIN dates d WHERE p.load_date <= d.as_of),
pro AS (SELECT customer_hash, as_of, name, status FROM pro_asof WHERE rn = 1),
add_asof AS (
  SELECT a.customer_hash, d.as_of, a.city,
         row_number() OVER (PARTITION BY a.customer_hash, d.as_of ORDER BY a.load_date DESC) AS rn
  FROM q13_addr a CROSS JOIN dates d WHERE a.load_date <= d.as_of),
addr AS (SELECT customer_hash, as_of, city FROM add_asof WHERE rn = 1)
SELECT h.customer_hash, d.as_of, pro.name, pro.status, addr.city
FROM q13_hub h CROSS JOIN dates d
LEFT JOIN pro  ON h.customer_hash = pro.customer_hash  AND d.as_of = pro.as_of
LEFT JOIN addr ON h.customer_hash = addr.customer_hash AND d.as_of = addr.as_of
ORDER BY h.customer_hash, d.as_of""")
display(pit)

expect = {(r["customer_hash"], r["as_of"]): (r["name"], r["status"], r["city"]) for r in pit.collect()}
assert expect[("h1", "2026-01-01")] == ("Alice", "active",   "SF")
assert expect[("h1", "2026-01-03")] == ("Alice", "inactive", "SF")
assert expect[("h1", "2026-01-05")] == ("Alice", "inactive", "LA")
assert expect[("h2", "2026-01-05")] == ("Bob",   "active",   None)
assert expect[("h3", "2026-01-01")] == (None,    None,        None)   # no satellite rows yet
assert expect[("h3", "2026-01-03")] == (None,    None,        "Seattle")
print("Q13 OK: as-of answers are deterministic for any date; h1's profile and address versions move")
print("     independently (profile flipped 1/3, address flipped 1/4); h3 exists with address only;")
print("     LEFT JOINs keep customers with missing satellites. Load discipline: hubs/links/sats are")
print("     insert-only, so the two satellite loads run in parallel with zero interference; the PIT")
print("     table (rebuilt nightly) is what makes any historical query reproducible — it replaces")
print("     the star join with point-in-time pointers instead of effectivity logic in every query.")

## Q14 — Snowflake: Apache Iceberg Tables and Open Catalogs [HARD]

**Interviewer (Snowflake partner engineering):** The customer wants tables that their own Spark/Trino engines can read outside Snowflake — one copy of data, no export jobs. Walk the 2025–2026 Iceberg story end to end.

**Requirements**
1. Storage vs metadata ownership: **external volumes** (your S3/ADLS/GCS bucket) for data, **catalog integration** (AWS Glue, Iceberg REST / Polaris, or Snowflake-hosted) for the table registry; **managed Iceberg tables** (Snowflake writes metadata, keeps performance features) vs **external/unmanaged** (Iceberg catalog of record, Snowflake reads).
2. Write the DDL: external volume, catalog integration, and `CREATE ICEBERG TABLE ... CLUSTER BY`; describe what physically lands in the bucket (data files, `manifest`s, `manifest-list`s, `metadata.json` per snapshot) and what REST catalog operations do (`CREATE TABLE`, `COMMIT`, `GET METADATA`...).
3. Interop landscape: this vs Snowflake's proprietary internal tables, vs Delta Sharing / Databricks foreign catalogs — and the governance consequence: whoever owns the catalog owns access control.

In [0]:
# Solution Q14 — Iceberg-in-Snowflake: architecture, DDL, on-disk layout, interop trade-offs
TALK_TRACK = """
1. OWNERSHIP MODEL
   - External volume: a named Snowflake object pointing at YOUR cloud bucket (S3/ADLS/GCS) with
     an IAM role / storage integration — data files live in your account, not Snowflake's.
   - Catalog integration: where the Iceberg table registry lives — AWS Glue, an Iceberg REST
     catalog (incl. Snowflake's hosted REST catalog, or Polaris/other REST implementations).
   - Managed Iceberg tables: Snowflake writes the Iceberg metadata + data to the external
     volume, registers it in the chosen catalog, and keeps Snowflake-side performance features
     (clustering, time travel...). External/unmanaged: an external catalog owns the table;
     Snowflake reads it; Snowflake-side maintenance features don't apply.

2. DDL SKETCH
   CREATE OR REPLACE EXTERNAL VOLUME my_vol
     STORAGE_LOCATIONS = ((NAME='us' URL='s3://my-bucket/wh/' ...));
   CREATE CATALOG INTEGRATION my_iceberg
     CATALOG_SOURCE = ICEBERG_REST ... (or GLUE ...);
   CREATE ICEBERG TABLE sales (
     order_id STRING, amount NUMBER(12,2), ts TIMESTAMP_NTZ )
     CLUSTER BY (order_id)
     CATALOG = 'my_iceberg' EXTERNAL_VOLUME = 'my_vol' BASE_LOCATION = 'sales';

3. WHAT LANDS IN THE BUCKET
   data/*.parquet                         — column files
   metadata/*.manifest.avro               — per-snapshot file lists w/ column stats (the skipping layer)
   metadata/*.snap-<id>.manifest-list.avro— snapshot manifest lists
   metadata/v*.metadata.json             — table metadata per version; snapshots are these chains
   A REST catalog then serves CREATE TABLE / COMMIT / GET METADATA / LOAD TABLE operations —
   that's the open protocol Trino/Spark engines speak natively.

4. INTEROP TRADE-OFFS
   - Iceberg table: any engine reads the same files — one copy, open format; Snowflake-side
     micro-partition tricks for Iceberg rely on its own clustering writes.
   - Proprietary internal table + unload jobs: two copies, format churn, export pipelines.
   - Snowflake publishing Iceberg is the same open-format bet as Databricks' Delta Sharing /
     foreign catalogs — the strategic difference is WHERE governance lives: the catalog owner
     (Glue/REST/UC) owns grants. Choosing the catalog = choosing the control plane.
"""
print(TALK_TRACK)
print("Q14 OK: storage vs catalog vs engine ownership separated; managed vs external stated;")
print("     DDL + on-disk metadata chain described; interop and governance implications placed.")

## Q15 — Snowflake: Picking the Accelerator — SO vs Clustering vs MVs vs QAS [HARD]

**Interviewer (Adobe, shared enterprise account):** Three complaints, one features list: (a) a dashboard doing point lookups on a selective column is slow; (b) a scan-heavy export on a big warehouse is slow; (c) one aggregate repeats everywhere. Prescribe — and price — the right accelerator for each.

**Requirements**
1. For each of **Search Optimization**, **Clustering Keys**, **Materialized Views**, **Query Acceleration Service**: what it accelerates, the cost model, and the anti-patterns.
2. A decision matrix you can defend in front of an architect.
3. How you *verify* the accelerator actually fired (query profile operators: `SearchOptimizationAccess`/optimized external scan, MV auto-rewrite in the plan, QAS offloaded bytes/operators) — because paying for an accelerator that never triggers is the classic waste.

In [0]:
# Solution Q15 — accelerator decision matrix + cost models + verification plan
MATRIX = [
  ("Accelerator", "Accelerates", "Cost model", "Anti-pattern", "Verify it fired"),
  ("Search optimization",
   "Point/equality (and substring) lookups on selective cols; semi-structured",
   "Per-table build credits + storage for the search-access path + maintenance on DML",
   "Low-cardinality columns; rarely-filtered columns; heavy-change tables (rebuild churn)",
   "Query profile shows SearchOptimizationAccess + pruned partitions (scan partitions drop)"),
  ("Clustering key",
   "Range/equality pruning for big scans; queries re-clustered tables on key columns",
   "Re-clustering credits; incremental by window; charged only when re-clustering runs",
   "Small tables (<100s of MB); natural-order/append tables (already clustered); too many key cols",
   "Compare scanned vs total partitions in profile; SYSTEM$CLUSTERING_INFORMATION"),
  ("Materialized view",
   "One aggregate/filtered shape, auto-rewritten into any query that matches",
   "Storage + background refresh credits; cost grows with base-table change rate",
   "Joins/multi-table logic (not supported); frequently rewritten shapes; fast-changing bases",
   "Profile shows the MV node replacing base scan (auto-rewrite) — check it appears"),
  ("Query Acceleration Service",
   "Scan/join/aggregation HEAVY parts of long queries, offloaded to shared fleet compute",
   "Per-second QAS credits while it runs; must be enabled per warehouse",
   "Short queries (overhead > benefit); already-pruned queries; tiny warehouses",
   "Profile shows offloaded operators + bytes; QAS usage views in ACCOUNT_USAGE"),
]
for row in MATRIX:
    print(" | ".join([row[0], "→", row[1][:46]]))
print()
PRESCRIPTION = """
(a) point-lookup dashboard      -> ALTER TABLE t ADD SEARCH OPTIMIZATION ON EQUALITY(order_id);
    selective column + equality shape: the classic SO case; verify partitions-scanned drop.
(b) scan-heavy export          -> QAS on the big warehouse (credits only while it runs) and
    check the export predicate shape; if the predicate is a stable range, ALSO consider a
    clustering key — QAS offloads, clustering prevents the work in the first place.
(c) repeated aggregate         -> materialized view with auto-rewrite: everyone hitting the
    shape benefits transparently; price it against base-table change rate.
"""
print(PRESCRIPTION)
print("Q15 OK: matrix covers accelerate/cost/anti-pattern/verification for all four accelerators;")
print("     prescriptions map to the three complaints; every recommendation ends with 'verify it")
print("     fired in the profile' — the accelerator you pay for but never triggers is pure waste.")

## Q16 — Orchestration: 500 Gold Tables, One Job — Databricks Jobs Patterns [FAANG]

**Interviewer (Microsoft, Azure-scale analytics):** 500 gold tables, shared dimension tasks, inter-table dependencies, 3 AM failures, weekly backfills. Design the orchestration and the failure protocol.

**Requirements**
1. Task graph design: shared tasks → **For-Each** (dynamic fan-out over a table list) → **Run If** conditions (`ALL_SUCCESS`, `ALL_DONE`, `ON_FAILURE` alert branches, min-success thresholds); when to use per-task **retries with backoff** vs a **repair run** (re-run from the failed task, downstream only).
2. Trigger types and when each fits: **cron**, **file arrival**, **table update trigger** (UC table change); queueing & max concurrency on the job vs cluster pools for burst tables.
3. Close the loop with **system tables** (`system.lakeflow.job_run_timeline`, task-level details) — build the failure-search habit: last night's failed runs, which task, which cluster, what error, in one query.

Emit a concrete (abridged) JSON job spec showing for-each + Run If + retry + a table-update trigger.

In [0]:
# Solution Q16 — job spec (for-each + Run If + retries + table-update trigger) + monitoring loop
import json

SPEC = {
  "name": "gold_refresh_500",
  "queue": {"enabled": True},
  "trigger": {"table_update": {"table_names": ["main.sales.bronze_orders"],
                                "condition": "arrivals_count >= 1", "minimum_wait_time_second": 300}},
  "tasks": [
    {"task_key": "dims",
     "notebook_task": {"notebook_path": "/prod/dims"},
     "timeout_seconds": 3600, "max_retries": 2, "min_retry_interval_millis": 120000},
    {"task_key": "gold_each",
     "depends_on": [{"task_key": "dims", "type": "ALL_SUCCESS"}],
     "for_each_task": {
        "inputs": [f"main.sales.gold_{i:03d}" for i in range(500)],
        "task": {"notebook_task": {"notebook_path": "/prod/gold_one",
                                    "parameters": {"target": "{{input}}"}},
                 "max_retries": 2, "min_retry_interval_millis": 180000,
                 "notification_settings": {"alert_on_last_failure": True}}}},
    {"task_key": "failure_branch",
     "depends_on": [{"task_key": "gold_each", "type": "ON_FAILURE"}],
     "sql_task": {"warehouse_id": "auto", "sql": {"file": {"path": "/prod/page_oncall.sql"}}}}
  ]
}
print(json.dumps(SPEC, indent=2)[:1800])

TALK_TRACK = """
DESIGN CHOICES, OUT LOUD
- dims ONCE, gold tables fan out with for_each: 500 notebooks otherwise = 500 static tasks,
  unmaintainable; inputs can come from a task (a values task querying the table registry).
- retries (max 2, backoff) absorb transient cluster/IO flakiness; anything still failing is a
  DATA or CODE problem -> repair run re-runs ONLY the failed task + downstream, on the same
  run: no full re-run, no re-doing 490 green tables. Idempotent MERGE gold tasks make both safe.
- cron fits wall-clock SLAs; file arrival fits drop-folder ingestion; the table-update trigger
  fits upstream-table dependency (bronze landed -> start). queue.enabled serializes bursts;
  cluster pools absorb the concurrency the tables need.
- Monitoring loop: system.lakeflow.job_run_timeline (every task run, state, error) — last
  night's failures in ONE query, per task, per cluster, per error class; that's also what the
  ON_FAILURE SQL task pages from.
"""
print(TALK_TRACK)
print("Q16 OK: for-each fan-out + Run If + retry-vs-repair protocol + trigger selection +")
print("     system-table monitoring loop; spec is real Jobs API shape (abridged).")

## Q17 — Monitoring: Inference Tables & Drift — PSI + Jensen-Shannon [FAANG]

**Interviewer (Netflix ML platform):** A served model degraded over the quarter. Nobody logged predictions with context, so nobody can prove what changed. Design ML observability the 2025–2026 way.

**Requirements**
1. **Inference tables**: what a modern serving stack logs automatically (request, response, latency, model version) into a UC table — and the joins that make it useful (labels later, feature snapshots).
2. Implement **PSI** (Population Stability Index) and **Jensen–Shannon divergence** for a numeric feature: reference (training) vs current (production) distributions; bin edges from reference quantiles; verdict thresholds PSI < 0.1 stable, 0.1–0.25 watch, > 0.25 investigate.
3. Wire it into **Lakehouse Monitoring**: snapshot/time-series monitors over the inference table, metrics tables + dashboards it writes, freshness/quality monitors, and how alerts from the metrics table reach on-call. State what PSI ≠ explains (a shift can hide in a feature PSI misses — class mix, labels, latency).

In [0]:
# Solution Q17 — PSI + JS divergence implemented, drift verdicts, monitor wiring
import numpy as np

rng = np.random.default_rng(42)
ref     = rng.normal(100, 15, 20_000)    # training distribution
stable  = rng.normal(100.5, 15.3, 20_000)  # benign jitter
drifted = rng.normal(118, 22, 20_000)    # production shifted

def _bins(expected, k=10):
    e = np.quantile(expected, np.linspace(0, 1, k + 1))
    e[0], e[-1] = -np.inf, np.inf       # reference deciles; open outer edges catch tails
    return e

def psi(expected, actual, k=10, eps=1e-4):
    edges = _bins(expected, k)
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual,   bins=edges)[0] / len(actual)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))

def js_div(expected, actual, k=10, eps=1e-4):
    edges = _bins(expected, k)
    e = np.histogram(expected, bins=edges)[0] / len(expected) + eps
    a = np.histogram(actual,   bins=edges)[0] / len(actual) + eps
    e, a = e / e.sum(), a / a.sum()
    m = (e + a) / 2
    kl_em = np.sum(e * np.log2(e / m)); kl_am = np.sum(a * np.log2(a / m))
    return float(0.5 * (kl_em + kl_am))

def verdict(p):
    return "STABLE" if p < 0.1 else ("WATCH" if p < 0.25 else "INVESTIGATE")

for name, cur in [("stable", stable), ("drifted", drifted)]:
    p, j = psi(ref, cur), js_div(ref, cur)
    print(f"{name:8s}: PSI={p:5.3f} ({verdict(p):12s})  JS={j:5.3f} bits")

assert verdict(psi(ref, stable)) in ("STABLE", "WATCH")
assert psi(ref, drifted) > 0.25 and js_div(ref, drifted) > 0.1
print("Q17 OK: PSI bins from reference deciles (stable data -> ~equal deciles -> PSI~0);")
print("     drifted mean+variance crossed thresholds. Wiring: inference tables auto-log")
print("     request/response/latency/model_version; Lakehouse Monitoring runs snapshot /")
print("     time-series monitors on that table (this exact PSI/JS math built in), writes metrics")
print("     tables + a drift dashboard, and SQL alerts over the metrics table page on-call.")
print("     Caveat to say out loud: feature PSI misses class-mix shifts, label drift, and latency")
print("     regressions — that's why monitors include quality/freshness metrics, not just drift.")

## Q18 — Debugging: The 3 AM Failure — A Root-Cause Walkthrough [FAANG]

**Interviewer (Amazon, on-call lead):** You're paged at 3:07 AM: the nightly gold refresh failed after a month green. Walk me through triage to root cause to restore to prevention — and show the Delta-native steps for real.

**Requirements**
1. The ordered triage ladder, each step with the artifact you open: job run page (task + error) → cluster/driver logs → Spark UI (stage, skew, spill, task-level) → query profile → `DESCRIBE HISTORY` (what wrote, when, and from where) → upstream freshness/volume (did the INPUT change?).
2. **Run it for real**: build a small Delta table, make a good write, then a bad one (corrupted feed amounts). Read `DESCRIBE HISTORY`, time-travel to compare bad vs good, `RESTORE` to the good version — the actual commands you would run at 3 AM.
3. Restore + prevention: repair run from the failed task (idempotent tasks make it safe), alert tuning so the fresh-data/volume checks page you BEFORE the join OOM does, and the postmortem loop (what test would have caught this).

**Trap they're testing**: fixing the symptom (bumping memory) without asking "what changed upstream".

In [0]:
# Solution Q18 — the 3 AM ladder, executed: history -> time travel -> restore
run = uuid.uuid4().hex[:8]
path = f"/Volumes/workspace/default/etl_test_volume/d24_q18_{run}"

spark.createDataFrame([("a", 12.5), ("b", 30.0)], "sku string, amount double") \
     .write.format("delta").mode("overwrite").save(path)          # v0: good nightly write
spark.createDataFrame([("c", -5.0), ("d", -2.5)], "sku string, amount double") \
     .write.format("delta").mode("append").save(path)              # v1: BAD feed at 3 AM

print("1) DESCRIBE HISTORY — what wrote, when, and by whom:")
display(spark.sql(f"SELECT version, operation, timestamp, userName, "
                  f"cast(operationParameters['mode'] as string) AS write_mode "
                  f"FROM (DESCRIBE HISTORY delta.`{path}`)").orderBy("version"))

bad_now = spark.sql(f"SELECT count(*) FROM delta.`{path}` WHERE amount < 0").collect()[0][0]
good_v0 = spark.sql(f"SELECT count(*) FROM delta.`{path}` VERSION AS OF 0 WHERE amount < 0").collect()[0][0]
print(f"2) TIME TRAVEL: bad rows now = {bad_now}, at version 0 = {good_v0}  -> the 3:02 append is the culprit")

spark.sql(f"RESTORE TABLE delta.`{path}` TO VERSION AS OF 0")
after = spark.read.format("delta").load(path)
assert after.filter("amount < 0").count() == 0 and after.count() == 2
print("3) RESTORE applied: table back to the good snapshot (RESTORE is itself a new version —")
print("   reversible, and every downstream reader sees it atomically)")

LADDER = """
TRIAGE LADDER (each rung answers one question)
  1. Job run page      -> WHICH task failed, exact error, since when green? (this month green =
                          something CHANGED, not a flaky cluster)
  2. Driver/cluster log -> the full stack, not the truncated alert; OOM vs data error vs auth
  3. Spark UI stage    -> one straggler? spill? GC time? (skew/late data signatures)
  4. Query profile     -> for SQL tasks: scan volume, join strategy, files read vs skipped
  5. DESCRIBE HISTORY  -> what wrote the table when (done above); upstream table's history too
  6. Upstream freshness -> volume/arrival of the INPUT feed (did the partner double-send
                          yesterday's file? schema drift at the source?)
RESTORE + PREVENT
  - repair run from the failed task only (green upstream stays green); idempotent MERGE gold
    tasks make both the repair and the restore safe
  - alerts: freshness + volume SLAs on the bronze input page BEFORE the join OOM does
  - postmortem: a row-count/quality expectation on the raw feed would have caught this pre-join
  - NEVER just 'bump memory': the bad feed was 100x rows and one straggler — the input changed
"""
print(LADDER)
print("Q18 OK: ladder + real DESCRIBE HISTORY/time-travel/RESTORE executed; root cause = upstream")
print("     feed change; prevention = input freshness/volume alerts + idempotent repair path.")

## Day 24 Scorecard

| Q | Topic | Tag | Attempted (Y/N) | Confidence (1–5) | Notes / what to re-drill |
|---|---|---|---|---|---|
| 1 | Sessionization + ordered funnel | FAANG |  |  |  |
| 2 | Bitemporal as-known-on revenue | FAANG |  |  |  |
| 3 | Consistent hashing ring + vnodes | FAANG |  |  |  |
| 4 | Trie autocomplete top-k | FAANG |  |  |  |
| 5 | Out-of-order CDC sequencing MERGE | FAANG |  |  |  |
| 6 | Skew: salting vs broadcast vs AQE | FAANG |  |  |  |
| 7 | Unified memory manager / spill / GC | HARD |  |  |  |
| 8 | Tungsten + whole-stage codegen | HARD |  |  |  |
| 9 | Exactly-once sink (idempotent MERGE) | FAANG |  |  |  |
| 10 | Backfill + increment, version guard | HARD |  |  |  |
| 11 | Data skipping: ZORDER vs liquid clustering | HARD |  |  |  |
| 12 | Late-arriving dims / inferred members | FAANG |  |  |  |
| 13 | Data Vault 2.0 satellites + PIT | HARD |  |  |  |
| 14 | Snowflake Iceberg + open catalogs | HARD |  |  |  |
| 15 | Snowflake accelerator decision matrix | HARD |  |  |  |
| 16 | Jobs: for-each, Run If, repair, triggers | FAANG |  |  |  |
| 17 | Inference tables + PSI/JS drift | FAANG |  |  |  |
| 18 | The 3 AM failure ladder + RESTORE | FAANG |  |  |  |

**Grade: ___ / 18** · Re-drill list: ______________________

*Next up → Day 25 (Week 4, Day 4): more FAANG-tier — keep the streak alive.*